<a href="https://colab.research.google.com/github/MalaikaJunaid/Seevia/blob/main/Aisle_Imagination_Similarity.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install necessary libraries for Zero-Shot and NLP
!pip install transformers torch pandas

In [4]:
from google.colab import userdata
from huggingface_hub import login

# Access your secret named HF_W_TOKEN
HF_TOKEN = userdata.get('HF_W_TOKEN')

if HF_TOKEN:
    login(HF_TOKEN)
    print("✅ Successfully logged in to Hugging Face!")
else:
    print("❌ Token 'HF_W_TOKEN' not found in Colab secrets. Please check the name.")

✅ Successfully logged in to Hugging Face!


In [5]:
from transformers import pipeline
import json

# 1. Load the comprehensive JSON we built
# Ensure you have uploaded 'save_mart_pwd_map.json' to Colab
with open('save_mart_pwd_map.json', 'r') as f:
    store_map = json.load(f)

# 2. Extract official Section Names as the "Target Classes"
# Research Note: These act as semantic anchors for the Zero-Shot model
candidate_labels = list(store_map['store_info']['sections'].values())

# 3. Initialize the Zero-Shot Pipeline
# We use BART-Large-MNLI as it is the industry standard for this task
classifier = pipeline("zero-shot-classification",
                      model="facebook/bart-large-mnli",
                      device=-1) # Uses CPU as requested

print("Seevia Zero-Shot Engine Initialized.")

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

Seevia Zero-Shot Engine Initialized.


In [3]:
def seevia_aisle_finder(query):
    # The Zero-Shot 'Magic': AI maps the query to the most relevant section
    result = classifier(query, candidate_labels)
    top_label = result['labels'][0]
    confidence = result['scores'][0]

    # Map the detected label back to the Section ID
    section_id = [k for k, v in store_map['store_info']['sections'].items() if v == top_label][0]

    # Logic to provide user guidance
    return {
        "User Query": query,
        "Detected Section": top_label,
        "Confidence Score": f"{confidence:.2%}",
        "Action": f"Head to {section_id} at Save Mart PWD."
    }

# 🧪 TEST CASES (Even for items not explicitly in your JSON)
print(seevia_aisle_finder("Mujhe bread par lagane wala shehad chahiye")) # Honey/Condiments
print(seevia_aisle_finder("Where is the floor cleaner?"))                # Cleaning
print(seevia_aisle_finder("Bacche ke pampers kahan hain?"))              # Baby Care

{'User Query': 'Mujhe bread par lagane wala shehad chahiye', 'Detected Section': 'Cereals, Tin Food, Sauces & Baking Items', 'Confidence Score': '49.24%', 'Action': 'Head to Section 3 at Save Mart PWD.'}
{'User Query': 'Where is the floor cleaner?', 'Detected Section': 'Tissues, Disposable, Room Freshners & Household Cleaning', 'Confidence Score': '43.89%', 'Action': 'Head to Section 6 at Save Mart PWD.'}
{'User Query': 'Bacche ke pampers kahan hain?', 'Detected Section': 'Baby Items, Sanitary, Napkin & Lotion', 'Confidence Score': '79.72%', 'Action': 'Head to Section 8 at Save Mart PWD.'}


In [7]:
import json

# Your comprehensive Save Mart PWD Map
save_mart_pwd_map = {
    "store_info": {
        "name": "Save Mart PWD",
        "location": "Islamabad (Basement)",
        "entry_points": [{"id": "main_entrance", "name": "Entrance Stairs", "nearby_aisles": ["Aisle 5", "Aisle 6"]}],
        "checkout_points": [
            {"id": "counter_3", "location": "In front of Aisle 1"},
            {"id": "counter_4", "location": "In front of Aisle 2"},
            {"id": "counter_2", "location": "Exclusively in front of Aisle 9 and 10"}
        ],
        "sections": {
            "S1": "Rice, Pulses, Frozen & Spices",
            "S2": "Pasta, Ketchup, Honey, Snacks & Drinks",
            "S3": "Cereals, Tin Food, Sauces & Baking Items",
            "S6": "Tissues, Disposable, Room Freshners & Household Cleaning",
            "S7": "Hair Care & Dental Care",
            "S8": "Baby Items, Sanitary, Napkin & Lotion"
        }
    }
}

with open('save_mart_pwd_map.json', 'w') as f:
    json.dump(save_mart_pwd_map, f, indent=4)

In [8]:
!pip install sentence-transformers
from sentence_transformers import SentenceTransformer, util
import torch

# 1. Load the Semantic Brain
model = SentenceTransformer('all-MiniLM-L6-v2')

# 2. Define the 'Target' labels from your Section names
aisle_categories = list(save_mart_pwd_map['store_info']['sections'].values())
aisle_embeddings = model.encode(aisle_categories, convert_to_tensor=True)

def seevia_imagine_aisle(user_query):
    # Vectorize the user's item (e.g., 'Surf' or 'Baisan')
    query_embedding = model.encode(user_query, convert_to_tensor=True)

    # Calculate Cosine Similarity
    cosine_scores = util.cos_sim(query_embedding, aisle_embeddings)

    # Find the Best Match
    best_match_idx = torch.argmax(cosine_scores).item()
    confidence = cosine_scores[0][best_match_idx].item()

    return {
        "Query": user_query,
        "Imagined Section": aisle_categories[best_match_idx],
        "Similarity Score": f"{confidence:.2f}"
    }

# 🧪 TEST: Even if these aren't in your JSON, it will 'Imagine' them correctly
print(seevia_imagine_aisle("Where is the detergent?"))  # Should match Section 6
print(seevia_imagine_aisle("Mujhe aata chahiye"))      # Should match Section 1

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


{'Query': 'Where is the detergent?', 'Imagined Section': 'Tissues, Disposable, Room Freshners & Household Cleaning', 'Similarity Score': '0.43'}
{'Query': 'Mujhe aata chahiye', 'Imagined Section': 'Rice, Pulses, Frozen & Spices', 'Similarity Score': '0.10'}
